# Corpus Quality Analysis (in-process)

Interactively run `Orchestrator.analyze_hierarchical` against the
labelled COVID-19 corpus in `fixtures/analyze_corpus.yaml`, using
the **live LLM** but **no FastAPI server**.

This complements `scripts/stress_analyze.py` (which exercises the
real REST path for load testing). Here the goal is to study a single
analysis end-to-end: inspect the synthesis, judge score, confidence,
and the deterministic `coding_trends` table.

### Prerequisites

- `LLM_API_KEY` set in the environment the same way the server reads it.
- `EMBEDDING_MODEL_PATH` pointing at a local BGE-M3 ONNX export — required
  for hierarchical mode. Without it, change `mode` below to `single_pass`
  and use a small `limit` (the corpus is well past the single-pass token
  cap, so single-pass will 413 above ~tens of records).


## 1. Imports and setup

In [ ]:
import sys
from datetime import UTC, datetime, timedelta
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

# Make ``scripts/`` importable so we can share the loader/sampler
# with stress_analyze.py.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "scripts"))

from stress_analyze import load_sample  # noqa: E402

from qfa.api.composition import build_orchestrator  # noqa: E402
from qfa.domain.models import AnalysisRequestModel, FeedbackRecordModel  # noqa: E402
from qfa.settings import AppSettings  # noqa: E402

## 2. Compose the orchestrator

`build_orchestrator` constructs the same domain object graph the
FastAPI lifespan builds, minus the `TrackingLLMAdapter` (no DB needed
for ad-hoc analysis). The first call also registers custom LiteLLM
model prices, so `result.cost` reflects real billing for any
self-hosted model in `qfa/resources/model_prices.yaml`.

In [ ]:
settings = AppSettings()
orchestrator = build_orchestrator(settings)
print(
    f"LLM:        {settings.llm.model}\n"
    f"Embedder:   {'configured' if orchestrator._embedder else 'NOT configured'}\n"
    f"Max tokens: {settings.llm.max_total_tokens:,}"
)

## 3. Sample a small batch

We're after a small-enough sample that the call returns in a couple
of minutes but large-enough that the hierarchical pipeline actually
chunks (i.e. exceeds the single-pass cap). 50 records is a good
starting point; bump it up to study how chunking and reduction scale.

In [ ]:
LIMIT = 50
SEED = 42

raw_records = load_sample(REPO_ROOT / "fixtures" / "analyze_corpus.yaml", LIMIT, SEED)
feedback_records = tuple(
    FeedbackRecordModel(id=r["id"], text=r["text"], metadata=r["metadata"])
    for r in raw_records
)
print(f"Sampled {len(feedback_records)} records (seed={SEED}).")
print(f"First record id: {feedback_records[0].id}")

## 4. Run `analyze_hierarchical`

The deadline is generous (5 minutes) because hierarchical mode
fires one LLM call per cluster plus a reduce call. With anonymization
enabled, Presidio is invoked before every LLM call.

_Note: this cell makes real LLM calls and costs real money._

In [ ]:
request = AnalysisRequestModel(
    feedback_records=feedback_records,
    prompt=(
        "Identify the main themes, sentiments, and any trending concerns "
        "in this feedback corpus. Highlight signals that change over time."
    ),
    tenant_id="notebook-quality-run",
    mode="hierarchical",
)
deadline = datetime.now(UTC) + timedelta(minutes=5)

result = await orchestrator.analyze_hierarchical(request, deadline)

## 5. Inspect the synthesis

`result.result` is the analyst-facing text (already de-anonymised);
`quality_score` and `uncertainty_explanation` come from the AI-as-judge
pass; `confidence` is the coverage-weighted mean of per-chunk judge
scores during the map phase.

In [ ]:
print(f"quality_score (judge):   {result.quality_score}")
print(f"confidence (map mean):   {result.confidence}")
print(f"uncertainty:             {result.uncertainty_explanation}\n")
display(Markdown(result.result))

## 6. Plot the coding-trend table

`coding_trends` is built deterministically from record metadata —
it does **not** depend on the LLM. We pivot it into a code × period
matrix and plot the codes with the strongest temporal variation.
The planted patterns in this corpus (spike, step, emerging,
declining) should be visible when the sample is large enough; with
only 50 records the signal is noisy.

In [ ]:
trends = result.coding_trends
if trends is None or not trends.cells:
    print("No coding_trends produced for this sample.")
else:
    df = pd.DataFrame(
        [{"code": c.code, "period": c.period, "count": c.count} for c in trends.cells]
    )
    pivot = df.pivot_table(
        index="code", columns="period", values="count", fill_value=0
    ).reindex(columns=list(trends.periods), fill_value=0)
    # Pick the most variable codes for legibility.
    top = pivot.loc[pivot.std(axis=1).nlargest(8).index]

    fig, ax = plt.subplots(figsize=(10, 5))
    top.T.plot(ax=ax, marker="o")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=45, ha="right")
    ax.set_ylabel("records")
    ax.set_title(f"coding_trends — top 8 codes by std (n={len(feedback_records)})")
    ax.legend(bbox_to_anchor=(1.04, 1), loc="upper left", fontsize=8)
    plt.tight_layout()
    plt.show()

## 7. (Optional) Tweak settings and re-run

Because the orchestrator is a plain Python object here, you can
construct a second one with different `analyze_settings` (e.g.
`min_cluster_size`) to see how chunking changes. Useful when
the synthesis above looks too granular or too coarse.

In [ ]:
# Example: smaller clusters → more chunks → more map calls (higher cost,
# potentially more nuance). Uncomment to try.
#
# from qfa.settings import AnalyzeSettings
#
# tweaked_settings = AppSettings()
# tweaked_settings.analyze = AnalyzeSettings(min_cluster_size=3)
# tweaked_orchestrator = build_orchestrator(tweaked_settings)
# deadline = datetime.now(UTC) + timedelta(minutes=5)
# alt = await tweaked_orchestrator.analyze_hierarchical(request, deadline)
# display(Markdown(alt.result))